**Objective:**
Build a conversational chatbot that can remember context and retrieve external information
during conversations.

In [8]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters \
    sentence-transformers faiss-cpu transformers accelerate streamlit

In [9]:
from google.colab import files
uploaded = files.upload()

import pandas as pd
csv_filename = list(uploaded.keys())[0]
df = pd.read_csv(csv_filename)
print("Dataset shape:", df.shape)
df.head()

Saving archive.zip to archive (1).zip
Dataset shape: (10, 4)


,ki_topic,ki_text,sample_question,sample_ground_truth
0,Setting Up a Mobile Device for Company Email,**Setting Up a Mobile Device for Company Email...,"""How do I set up my company email on my mobile...",To set up your company email on your mobile de...
1,Resetting a Forgotten PIN,**Resetting a Forgotten PIN**\n\nIf you have f...,"I forgot my PIN, how can I reset it?","Don't worry, I'm here to help To reset your fo..."
2,Configuring VPN Access for Remote Workers,**Configuring VPN Access for Remote Workers**\...,How do I set up VPN access on my laptop so I c...,To set up VPN access on your laptop and access...
3,Troubleshooting Issues with Microsoft Office,**Troubleshooting Issues with Microsoft Office...,"""My Microsoft Word keeps freezing every time I...",I'd be happy to help you troubleshoot the issu...
4,Setting Up a Conference Call on Cisco Webex,"To set up a conference call on Cisco Webex, fo...",How do I set up a conference call on Cisco Web...,To set up a conference call on Cisco Webex wit...


**Text Column Auto-Detect & Document Build**

In [10]:
text_col = max(df.columns, key=lambda c: df[c].astype(str).str.len().mean())
print("Detected text column:", text_col)

df = df.dropna(subset=[text_col]).reset_index(drop=True)

documents_text = []
for idx, row in df.iterrows():
    documents_text.append({"title": f"doc_{idx}", "content": str(row[text_col])})

print("Total documents:", len(documents_text))

Detected text column: ki_text
Total documents: 10


**Chunking**

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

all_chunks = []
for doc in documents_text:
    for chunk in splitter.split_text(doc["content"]):
        all_chunks.append(Document(page_content=chunk, metadata={"source": doc["title"]}))

print("Total chunks:", len(all_chunks))

Total chunks: 73


**Embeddings + Vector Store**

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(all_chunks, embedding_model)
print("Vector store size:", vectorstore.index.ntotal)

vectorstore.save_local("faiss_index")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store size: 73


**Load LLM**

In [13]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded:", model_name)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-base


**Custom LangChain LLM Wrapper**

In [14]:
from langchain_core.language_models.llms import LLM
from typing import Optional, List, Any

class FlanT5LLM(LLM):
    """Direct wrapper - bypasses transformers pipeline() entirely (v5-compatible)."""
    model: Any = None
    tokenizer: Any = None
    max_new_tokens: int = 256

    @property
    def _llm_type(self) -> str:
        return "flan_t5_custom"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
        with torch.no_grad():
            output_ids = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        return self.tokenizer.decode(output_ids[0], skip_special_tokens=True)

llm = FlanT5LLM(model=seq2seq_model, tokenizer=tokenizer)

# quick test
print(llm.invoke("question: What is the capital of France? context: France is a country in Europe. Its capital is Paris."))

Paris


**RAG and Conversational Memory**

In [15]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
chat_history = []

def ask(question, history_window=3):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in retrieved_docs)

    history_text = ""
    for turn in chat_history[-history_window:]:
        history_text += f"User: {turn['question']}\nAssistant: {turn['answer']}\n"

    prompt = f"""Answer the question using the context and conversation history below.
If the answer isn't in the context, say you don't know.

Context:
{context}

Conversation history:
{history_text}

Question: {question}
Answer:"""

    answer = llm.invoke(prompt)
    chat_history.append({"question": question, "answer": answer})

    return {"answer": answer, "sources": [d.metadata["source"] for d in retrieved_docs]}

 Test Multi-turn, Context-Aware

In [16]:
r1 = ask("What information is in this dataset?")
print("Q1:", r1["answer"])
print("Sources:", r1["sources"])

r2 = ask("Can you tell me more about that?")
print("\nQ2:", r2["answer"])

Q1: Incoming server: [company email server or personal email server]
Sources: ['doc_5', 'doc_2', 'doc_9']

Q2: This dataset contains information about the number of users who have a personal email account.


**Streamlit App File**

In [27]:
app_code = '''
import streamlit as st
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

st.set_page_config(page_title="RAG Chatbot", page_icon="🤖")
st.title("🤖 Context-Aware RAG Chatbot")

@st.cache_resource
def load_components():
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)
    tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    return vectorstore, tokenizer, model

vectorstore, tokenizer, model = load_components()
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def generate_answer(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=256)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

if "messages" not in st.session_state:
    st.session_state.messages = []
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

user_input = st.chat_input("Ask a question...")
if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    retrieved_docs = retriever.invoke(user_input)
    context = "\\n\\n".join(d.page_content for d in retrieved_docs)

    history_text = ""
    for turn in st.session_state.chat_history[-3:]:
        q = turn["question"]
        a = turn["answer"]
        history_text += f"User: {q}\\nAssistant: {a}\\n"

    prompt = f"""Answer using the context and history below.

Context:
{context}

History:
{history_text}

Question: {user_input}
Answer:"""

    answer = generate_answer(prompt)
    st.session_state.chat_history.append({"question": user_input, "answer": answer})

    with st.chat_message("assistant"):
        st.write(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created")

app.py created


**Run **

In [28]:
!pip install -q pyngrok
from pyngrok import ngrok

ngrok.set_auth_token("3FkrJJdH19tfJsmkxToakWHd8JG_64GXipZcmDNrM33n6R3sw")
!streamlit run app.py &>/content/logs.txt &
public_url = ngrok.connect(8501)
print("Live at:", public_url)

Live at: NgrokTunnel: "https://unlaced-jujitsu-cosigner.ngrok-free.dev" -> "http://localhost:8501"
